## Foundation — LLM + Tool Binding (NOT an Agent)

### What is this?
Before learning agents, you need to understand the building block: **LLM + Tools**.
This is the manual approach — you write code that acts as the loop between the LLM and tools.
You are the agent. No framework manages this for you.

### How it works
```
bind_tools → tells the LLM what tools exist (via docstrings)

LLM does NOT call tools. It outputs a structured tool_call object.
YOUR code reads it and executes the function.
YOUR code sends the result back.
YOUR code calls the LLM again for the final answer.
```

### Key insight
```
response = llm_with_tools.invoke('What is 144/12?')

response.content     = ''                ← LLM didn't answer, it chose to call a tool
response.tool_calls  = [{'name': 'calculate', 'args': {'expression': '144/12'}, 'id': '...'}]
                                         ← LLM output is a PLAN, not an action
```

### Why this is NOT an agent
An agent manages its own loop — it decides when to call tools, executes them, and knows
when it has enough information to stop. Here YOU manage every step manually.

### What comes next
Notebook 2 (ReAct Agent) and Notebook 3 (Tool Calling Agent) show how agent frameworks
take over this manual loop — you just give the query and get the answer.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries
from langchain_core.messages import HumanMessage, ToolMessage

llm = get_llm()
print(f'LLM ready | Tools available: {[t.name for t in TOOLS]}')

### Step 1 — bind_tools: Register Tools With the LLM
`bind_tools` sends tool descriptions (name, docstring, parameters) to the LLM
so it knows what tools are available. The LLM does NOT execute them — it just knows about them.

In [ ]:
llm_with_tools = llm.bind_tools(TOOLS)

# What does the LLM output when it wants to use a tool?
response = llm_with_tools.invoke('What is 144 divided by 12?')

print(f'response.content    = "{response.content}"')      # empty — LLM chose a tool
print(f'response.tool_calls = {response.tool_calls}')    # LLM output: PLAN to call calculate
print()
print('Notice: the LLM did NOT compute 12. It output a tool call instruction.')
print('YOUR code must now read this and actually run calculate("144/12").')

### Step 2 — YOU Execute the Tool (Manual Loop)
Read the tool call, run the function, wrap the result in `ToolMessage`, send back to LLM.
You are writing what an agent framework would do automatically.

In [ ]:
tool_map = {t.name: t for t in TOOLS}

query    = 'What is 144 divided by 12?'
messages = [HumanMessage(content=query)]

# Round 1: LLM decides which tool to call
response = llm_with_tools.invoke(messages)
messages.append(response)
print(f'Step 1 — LLM says: call {response.tool_calls[0]["name"]}({response.tool_calls[0]["args"]})')

# Round 2: YOUR code runs the tool
for call in response.tool_calls:
    result = tool_map[call['name']].invoke(call['args'])
    print(f'Step 2 — YOU run:  {call["name"]}({call["args"]}) → {result}')
    messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))

# Round 3: LLM sees tool result and forms final answer
final = llm_with_tools.invoke(messages)
print(f'Step 3 — LLM says: {final.content}')
print()
print('You manually did 3 steps. An agent framework does this automatically.')

### Step 3 — Direct Answer (No Tool Called)
When the LLM can answer from training data, it skips tools and answers directly.
In this case `response.tool_calls` is empty.

In [ ]:
r = llm_with_tools.invoke('What does LCEL stand for in LangChain?')
print(f'tool_calls: {r.tool_calls}')   # empty — LLM answered directly
print(f'content:    {r.content}')

### Step 4 — Multi-Tool Case
The LLM can request multiple tools in one response. Your code must loop through all of them.

In [ ]:
messages = [HumanMessage(content='Search docs for RAG and also calculate 25 multiplied by 4')]
r = llm_with_tools.invoke(messages)
messages.append(r)

print(f'LLM requested {len(r.tool_calls)} tool(s):')
for call in r.tool_calls:
    result = tool_map[call['name']].invoke(call['args'])
    print(f'  {call["name"]}({call["args"]}) → {str(result)[:80]}')
    messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))

final = llm_with_tools.invoke(messages)
print(f'\nFinal answer: {final.content}')

### Full Function — Manual LLM + Tools Loop
This is the complete pattern you'd write without an agent framework.

In [ ]:
def llm_with_tools_call(query: str) -> str:
    messages = [HumanMessage(content=query)]
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if response.tool_calls:
        for call in response.tool_calls:
            result = tool_map[call['name']].invoke(call['args'])
            messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
        final = llm_with_tools.invoke(messages)
        return final.content

    return response.content

### Test All Queries

In [ ]:
run_queries(llm_with_tools_call, label='Foundation — LLM + Tool Binding (Manual)')

### Summary — What's Missing Here?

```
Problem 1: YOU write the loop — if the LLM needs to call 5 tools in sequence, you
           need to handle every iteration manually.

Problem 2: No reasoning trace — you see what the LLM decided but not WHY.

Problem 3: No autonomous decision-making — the LLM cannot decide 'I need one more
           tool call' after seeing a result unless you code that logic.

→ Notebook 2 (ReAct Agent): agent writes its reasoning before every action.
→ Notebook 3 (Tool Calling Agent): create_agent manages the loop so you don't have to.
```